# 1. Subsample MODI-HChar
Copies up to `PER_CLASS` random images per class into a flat `dst/<class>/*.png` folder. Works for both `src/<class>/` and `src/<group>/<class>/` (digits / consonants / vowels).

In [ ]:
import os, sys
from pathlib import Path

# run everything from the project root, whether Jupyter was started there or in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))  # so `backend.ml.preprocessing` can be imported
print("project root:", ROOT)

In [ ]:
# ---- settings ----
SRC = "data/raw/modi_hchar_full"
DST = "data/processed/offline_subset"
PER_CLASS = 300      # keep the same value in notebooks 02 and 03
MAX_CLASSES = 20     # 0 = all 57 classes
CLASSES = None       # or a list of class names to keep
SEED = 42

In [ ]:
import random
import shutil
from collections import Counter

EXTS = {".png", ".jpg", ".jpeg", ".bmp"}


def has_images(d):
    return any(f.suffix.lower() in EXTS for f in d.iterdir() if f.is_file())


def discover(src):
    """{class_name: folder} for the flat or grouped layout."""
    found = []  # (group, class_name, folder)
    for d in sorted(p for p in src.iterdir() if p.is_dir()):
        if has_images(d):
            found.append(("", d.name, d))
        else:
            for sub in sorted(p for p in d.iterdir() if p.is_dir()):
                if has_images(sub):
                    found.append((d.name, sub.name, sub))
    counts = Counter(name for _, name, _ in found)
    return {(f"{g}_{n}" if g and counts[n] > 1 else n): folder for g, n, folder in found}

In [ ]:
rng = random.Random(SEED)
src, dst = Path(SRC), Path(DST)
folders = discover(src)
assert folders, f"no folders containing images found under {src}"
classes = sorted(folders)
print(f"found {len(classes)} classes under {src}")
if CLASSES:
    classes = [c for c in classes if c in set(CLASSES)]
elif MAX_CLASSES:
    classes = sorted(rng.sample(classes, min(MAX_CLASSES, len(classes))))

total = 0
for c in classes:
    files = sorted(f for f in folders[c].iterdir() if f.suffix.lower() in EXTS)
    rng.shuffle(files)
    files = files[:PER_CLASS] if PER_CLASS else files
    (dst / c).mkdir(parents=True, exist_ok=True)
    for f in files:
        shutil.copy2(f, dst / c / f.name)
    total += len(files)
print(f"copied {total} images across {len(classes)} classes to {dst}")